# 需要予測① — ベースライン：素朴な物差しをテストで測る

出典：第04章 02節／パス：`04_demand/01_baseline.ipynb`／パッケージ：numpy・pandas・scikit-learn／実行時間の目安：約1分

このノート単体で実行できる。月次航空旅客数（公開データ）の取得に失敗したときは `input/passengers.csv`、さらに欠品のときは内蔵生成へ切り替わる。
改善モデルのコードは `02_improved.ipynb` に分けてある。ホールドアウトは直近（未来側）に取り、最終評価はテストで1回だけ行う。


In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder

AIR_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"


def make_demand_series(n: int = 144, seed: int = 4) -> pd.DataFrame:
    # 公開データと同じ2列形式の合成代替（取得失敗時のみ使う手順確認用）
    r = np.random.default_rng(seed)
    t = np.arange(n)
    month = t % 12
    seas = 60.0 * np.sin(2 * np.pi * (month - 3) / 12.0) + 30.0 * (month == 6) + 40.0 * (month == 11)
    y = 120.0 + 1.1 * t + seas + r.normal(0, 12.0, size=n)
    dates = pd.date_range("1949-01", periods=n, freq="MS")
    return pd.DataFrame({"date": dates, "value": y})


try:
    raw = pd.read_csv(AIR_URL)
    raw.columns = ["date", "value"]
    raw["date"] = pd.to_datetime(raw["date"])
    ts = raw.sort_values("date").reset_index(drop=True)
    print("公開データを使用: 月次航空旅客数（1949-1960）")
except Exception:
    try:
        ts = pd.read_csv("input/passengers.csv", parse_dates=["date"])
        print("input/passengers.csv（合成代替）を使用")
    except FileNotFoundError:
        ts = make_demand_series()
        print("合成系列を生成（手順確認用）")
print(f"期間: {ts['date'].min().date()}〜{ts['date'].max().date()} / {len(ts)}か月")


公開データを使用: 月次航空旅客数（1949-1960）
期間: 1949-01-01〜1960-12-01 / 144か月


In [2]:
# 特徴量は予測時点で確定するものだけ：前月・前年同月・月・trend
df = ts.copy()
df["lag1"] = df["value"].shift(1)
df["lag12"] = df["value"].shift(12)
df["trend"] = np.arange(len(df))
df["month"] = df["date"].dt.month.astype(str)
df = df.dropna().reset_index(drop=True)
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
M = enc.fit_transform(df[["month"]])
X = np.column_stack([df[["lag1", "lag12", "trend"]].to_numpy(), M])
yv = df["value"].to_numpy()
# 直近24か月をホールドアウト（未来の評価用に残す）
X_tr, X_te, y_tr, y_te = X[:-24], X[-24:], yv[:-24], yv[-24:]
print(f"学習: {len(X_tr)}か月 / テスト: {len(X_te)}か月")

学習: 108か月 / テスト: 24か月


In [3]:
# ベースライン：前年同月と前月の物差し（最終評価はテストで1回だけ）
lag12_te = df["lag12"].to_numpy()[-24:]
lag1_te = df["lag1"].to_numpy()[-24:]
rmse_season = float(np.sqrt(mean_squared_error(y_te, lag12_te)))
rmse_naive = float(np.sqrt(mean_squared_error(y_te, lag1_te)))
print(f"[前年同月] RMSE={rmse_season:.2f}")
print(f"[前月] RMSE={rmse_naive:.2f}")


[前年同月] RMSE=49.99
[前月] RMSE=51.78


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
plt.figure()
plt.plot(y_te, label="actual")
plt.plot(df["lag12"].to_numpy()[-24:], label="seasonal-naive")
plt.xlabel("month (test)"); plt.ylabel("passengers"); plt.legend(); plt.title("actual vs baseline (test)")
plt.savefig(f"{FIG}/code-result-j04-01.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j04-01.png")


saved code-result-j04-01.png


## 読み取りどころ

- 季節ナイーブが意外に強いこと（季節性が支配的な系列の常）。
- この2つがベースラインの物差しになること（改善幅は `02_improved.ipynb` で測る）。
- ホールドアウトは直近（未来側）に取ること。ランダム分割はしない（改善モデルのファイルへ）。
